# 04 — Endogenous Forestry (Manley Equation)

This notebook demonstrates NZUpy's endogenous forestry mode, in which new afforestation responds to carbon prices via the Manley logistic equation.

In the default **exogenous** mode, forestry supply is a fixed time-series from input data. In **endogenous** mode, the carbon price path drives new planting decisions, which in turn affect supply — creating a feedback loop within the optimisation.

**Key concepts:**
- The Manley logistic equation determines new planting area each year based on the NPV-weighted forward carbon price versus the land opportunity cost
- Historic forests (planted pre-2023) contribute fixed exogenous supply from `historical_removals.csv`
- New forests (planted 2023+) are modelled dynamically via vintage convolution with yield curves
- Sensitivities (`low` / `central` / `high`) vary the responsiveness of afforestation in the logistic equation

In [ ]:
# Standard imports
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from pathlib import Path
import sys
import os

# Project root on path
project_root = Path().absolute().parent
sys.path.insert(0, str(project_root))

from model.core.base_model import NZUpy

data_dir = project_root / "data"
output_dir = project_root / "examples" / "outputs" / "04_endogenous_forestry"
os.makedirs(output_dir, exist_ok=True)

## 1. Exogenous baseline run

First we run the model in its default exogenous mode. This gives us the reference carbon price and forestry supply to compare against.

In [ ]:
NZU_exog = NZUpy(data_dir=data_dir)
NZU_exog.define_time(2024, 2050)
NZU_exog.define_scenarios(["Exogenous"])
NZU_exog.allocate()
NZU_exog.fill_defaults()
NZU_exog.fill_component("stockpile", config="EY24_central")
NZU_exog.run()

exog_prices   = NZU_exog.prices.xs("carbon_price", level="variable", axis=1)["Exogenous"]
exog_forestry = NZU_exog.forestry.xs("removals", level="variable", axis=1)["Exogenous"]
exog_demand   = NZU_exog.demand.xs("emissions", level="variable", axis=1)["Exogenous"]

print(f"Exogenous price 2024: ${exog_prices[2024]:.2f} -> 2050: ${exog_prices[2050]:.2f}")
print(f"Exogenous forestry 2024: {exog_forestry[2024]:,.0f} kt -> 2050: {exog_forestry[2050]:,.0f} kt")
print(f"Exogenous demand 2050: {exog_demand[2050]:,.0f} kt")

## 2. Endogenous run — central sensitivity

Enable the Manley equation with `fill('forestry_mode', 'endogenous')`. The `manley_sensitivity` controls how much land is available (f):

| Sensitivity | f (max available land) | LMV (land market value) |
|-------------|------------------------|-------------------------|
| `low`       | 60,000 ha              | $10,000/ha              |
| `central`   | 100,000 ha             | $10,000/ha              |
| `high`      | 120,000 ha             | $7,500/ha               |

In [ ]:
NZU_endog = NZUpy(data_dir=data_dir)
NZU_endog.define_time(2024, 2050)
NZU_endog.define_scenarios(["Endogenous"])
NZU_endog.allocate()
NZU_endog.fill_defaults()
NZU_endog.fill_component("stockpile", config="EY24_central")

# Enable endogenous forestry
NZU_endog.fill("forestry_mode", "endogenous")
NZU_endog.fill("manley_sensitivity", "central") # f = 100,000 ha
NZU_endog.fill("forestry_price_assumption", "future") # NPV-weighted forward price (default)

NZU_endog.run()

endog_prices  = NZU_endog.prices.xs("carbon_price", level="variable", axis=1)["Endogenous"]
endog_total   = NZU_endog.forestry.xs("removals", level="variable", axis=1)["Endogenous"]
endog_hist    = NZU_endog.forestry.xs("historic_supply",level="variable", axis=1)["Endogenous"]
endog_manley  = NZU_endog.forestry.xs("manley_supply", level="variable", axis=1)["Endogenous"]
endog_planting= NZU_endog.forestry.xs("manley_planting_total", level="variable", axis=1)["Endogenous"]
endog_fwd_px  = NZU_endog.forestry.xs("manley_price", level="variable", axis=1)["Endogenous"]

print(f"Endogenous price 2024: ${endog_prices[2024]:.2f} -> 2050: ${endog_prices[2050]:.2f}")
print(f"Total forestry 2050: {endog_total[2050]:,.0f} kt (historic: {endog_hist[2050]:,.0f} + Manley new: {endog_manley[2050]:,.0f})")
print(f"Manley planting rate (steady state): {endog_planting[2030]:,.0f} ha/yr")
print(f"Manley forward price (2050, 2021$): ${endog_fwd_px[2050]:.2f}")

## 3. Diagnostic outputs

The endogenous forestry mode exposes several diagnostic variables:

- **`historic_supply`** — supply from old forests (pre-2023 planting) — declines over time
- **`manley_supply`** — supply from new forests planted by the Manley equation — grows as vintages accumulate
- **`manley_price`** — the forward price (2021 NZD/tCO2-e) fed into the logistic equation
- **`manley_planting_total`** — new afforestation area per year (ha)

In [ ]:
diagnostics = pd.DataFrame({
    "price ($/tCO2)":       endog_prices.loc[2024:2050].round(2),
    "fwd_price (2021$)":    endog_fwd_px.loc[2024:2050].round(2),
    "planting (ha)":        endog_planting.loc[2024:2050].round(0),
    "manley_supply (kt)":   endog_manley.loc[2024:2050].round(1),
    "historic_supply (kt)": endog_hist.loc[2024:2050].round(1),
    "total_forestry (kt)":  endog_total.loc[2024:2050].round(1),
    "demand (kt)":          NZU_endog.demand.xs("emissions", level="variable", axis=1)["Endogenous"].loc[2024:2050].round(1),
})
print(diagnostics.to_string())

## 4. Sensitivity comparison (low / central / high)

We run all three sensitivities to see how the available land assumption affects carbon prices and forestry supply.

In [ ]:
results = {}
for sens in ["low", "central", "high"]:
    nzu = NZUpy(data_dir=data_dir)
    nzu.define_time(2024, 2050)
    nzu.define_scenarios([sens])
    nzu.allocate()
    nzu.fill_defaults()
    nzu.fill_component("stockpile", config="EY24_central")
    nzu.fill("forestry_mode", "endogenous")
    nzu.fill("manley_sensitivity", sens)
    nzu.run()
    results[sens] = {
        "prices":   nzu.prices.xs("carbon_price", level="variable", axis=1)[sens],
        "forestry": nzu.forestry.xs("removals",   level="variable", axis=1)[sens],
        "planting": nzu.forestry.xs("manley_planting_total", level="variable", axis=1)[sens],
    }
    print(f"{sens}: price_2050=${results[sens]['prices'][2050]:.1f}  forestry_2050={results[sens]['forestry'][2050]:,.0f} kt  planting_2030={results[sens]['planting'][2030]:,.0f} ha/yr")

## 5. Chart — carbon price comparison

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=exog_prices.loc[2024:2050].index, y=exog_prices.loc[2024:2050].values,
    name="Exogenous", line=dict(dash="dash", color="grey", width=2)
))
for sens, color in zip(["low", "central", "high"], ["#2196F3", "#4CAF50", "#FF9800"]):
    prices = results[sens]["prices"].loc[2024:2050]
    fig.add_trace(go.Scatter(x=prices.index, y=prices.values,
                             name=f"Endogenous ({sens})", line=dict(color=color, width=2)))

fig.update_layout(
    title="Carbon Price: Exogenous vs Endogenous Forestry",
    xaxis_title="Year", yaxis_title="$/tCO2-e (2023 NZD)",
    legend_title="Scenario", template="plotly_white"
)
fig.show()

## 6. Chart — forestry supply comparison

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=exog_forestry.loc[2024:2050].index, y=exog_forestry.loc[2024:2050].values,
    name="Exogenous", line=dict(dash="dash", color="grey", width=2)
))
for sens, color in zip(["low", "central", "high"], ["#2196F3", "#4CAF50", "#FF9800"]):
    fs = results[sens]["forestry"].loc[2024:2050]
    fig.add_trace(go.Scatter(x=fs.index, y=fs.values,
                             name=f"Endogenous ({sens})", line=dict(color=color, width=2)))

fig.add_trace(go.Scatter(
    x=exog_demand.loc[2024:2050].index, y=exog_demand.loc[2024:2050].values,
    name="Demand (reference)", line=dict(dash="dot", color="black", width=1.5)
))

fig.update_layout(
    title="Forestry Supply: Exogenous vs Endogenous",
    xaxis_title="Year", yaxis_title="kt CO2-e",
    legend_title="Scenario", template="plotly_white"
)
fig.show()

## 7. Chart — Manley planting areas by sensitivity

In [ ]:
f_labels = {"low": "60,000", "central": "100,000", "high": "120,000"}
colors   = {"low": "#2196F3", "central": "#4CAF50", "high": "#FF9800"}

fig = go.Figure()
for sens in ["low", "central", "high"]:
    planting = results[sens]["planting"].loc[2024:2050]
    fig.add_trace(go.Scatter(
        x=planting.index, y=planting.values,
        name=f"{sens.capitalize()} (f={f_labels[sens]} ha)",
        line=dict(color=colors[sens], width=2)
    ))

fig.update_layout(
    title="Manley New Afforestation: Annual Planting Rate",
    xaxis_title="Year", yaxis_title="Hectares per year",
    legend_title="Sensitivity", template="plotly_white"
)
fig.show()

## 8. Customising individual Manley parameters

Advanced users can override individual Manley parameters directly via `fill()`:

In [ ]:
# Example: higher discount rate and lagged current price instead of NPV forward price
NZU_custom = NZUpy(data_dir=data_dir)
NZU_custom.define_time(2024, 2050)
NZU_custom.define_scenarios(["Custom"])
NZU_custom.allocate()
NZU_custom.fill_defaults()
NZU_custom.fill_component("stockpile", config="EY24_central")
NZU_custom.fill("forestry_mode", "endogenous")
NZU_custom.fill("manley_sensitivity", "central")
NZU_custom.fill("forestry_discount_rate", 0.10)         # Higher discount rate -> lower forward price
NZU_custom.fill("forestry_price_assumption", "current")    # Lagged current price instead of NPV
NZU_custom.run()

custom_prices = NZU_custom.prices.xs("carbon_price", level="variable", axis=1)["Custom"]
custom_plant  = NZU_custom.forestry.xs("manley_planting_total", level="variable", axis=1)["Custom"]

print("Custom (lagged current price, 10% discount rate):")
print(f"  Price 2050: ${custom_prices[2050]:.2f}")
print(f"  Planting 2030: {custom_plant[2030]:,.0f} ha/yr")
print()
print("Central (forward price, 8% discount rate):")
print(f"  Price 2050: ${endog_prices[2050]:.2f}")
print(f"  Planting 2030: {endog_planting[2030]:,.0f} ha/yr")

## Summary

The Manley equation introduces a price-responsive feedback into the NZ ETS model:

1. The optimiser tests different price-change rates
2. For each candidate price path, the Manley equation computes how much new forest gets planted (based on the NPV-weighted forward price vs land cost)
3. Those forests mature over time via yield-curve convolution, adding supply
4. The optimiser selects the price path that minimises the cumulative supply-demand gap

The endogenous model therefore finds a self-consistent equilibrium where carbon prices, planting incentives, and forestry supply are all mutually determined.

**Key observations:**
- Higher sensitivities (more available land) produce more planting and more forestry supply
- The transition gap in early years (historic supply < full exogenous supply) means endogenous prices are typically higher than exogenous in the first decade
- By the 2040s, accumulated Manley new-forest supply typically exceeds what the exogenous scenario assumed